In [0]:
dbutils.secrets.get(scope="tokyo-olympic", key="client-id")

'[REDACTED]'

In [0]:
df = spark.read.option("header", "true").csv(
    "abfss://tokyo-olympic-data@tokyoolympicdatasohom.dfs.core.windows.net/raw-data/athletes.csv"
)
df.show(5)

+-----------------+------+-------------------+
|             Name|   NOC|         Discipline|
+-----------------+------+-------------------+
|  AALERUD Katrine|Norway|       Cycling Road|
|      ABAD Nestor| Spain|Artistic Gymnastics|
|ABAGNALE Giovanni| Italy|             Rowing|
|   ABALDE Alberto| Spain|         Basketball|
|    ABALDE Tamara| Spain|         Basketball|
+-----------------+------+-------------------+
only showing top 5 rows


In [0]:
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim, length

In [0]:
athletes = (
    spark.read.option("header", "true")
    .option("inferSchema", "true")
    .csv(
        "abfss://tokyo-olympic-data@tokyoolympicdatasohom.dfs.core.windows.net/raw-data/athletes.csv"
    )
)
coaches = (
    spark.read.option("header", "true")
    .option("inferSchema", "true")
    .csv(
        "abfss://tokyo-olympic-data@tokyoolympicdatasohom.dfs.core.windows.net/raw-data/coaches.csv"
    )
)
teams = (
    spark.read.option("header", "true")
    .option("inferSchema", "true")
    .csv(
        "abfss://tokyo-olympic-data@tokyoolympicdatasohom.dfs.core.windows.net/raw-data/teams.csv"
    )
)
entriesgender = (
    spark.read.option("header", "true")
    .option("inferSchema", "true")
    .csv(
        "abfss://tokyo-olympic-data@tokyoolympicdatasohom.dfs.core.windows.net/raw-data/entriesgender.csv"
    )
)
medals = (
    spark.read.option("header", "true")
    .option("inferSchema", "true")
    .csv(
        "abfss://tokyo-olympic-data@tokyoolympicdatasohom.dfs.core.windows.net/raw-data/medals.csv"
    )
)

# Data Transformation

### 1. Inconsistent Spaces


In [0]:
athletes.select(
    col("Name").alias("original"),
    length(col("Name")).alias("original_length"),
    trim(col("Name")).alias("trimmed"),
    length(trim(col("Name"))).alias("trimmed_length"),
).filter(
    length(col("Name")) != length(trim(col("Name")))
).show(truncate=False)

+----------------+---------------+---------------+--------------+
|original        |original_length|trimmed        |trimmed_length|
+----------------+---------------+---------------+--------------+
|GIRAUD Aurelien |16             |GIRAUD Aurelien|15            |
+----------------+---------------+---------------+--------------+



In [0]:
athletes.select(
    col("NOC").alias("Original"),
    length(col("NOC")).alias("Original_Length"),
    trim(col("NOC")).alias("Trimmed"),
    length(trim(col("NOC"))).alias("Trimmed_length"),
).filter(
    length(col("NOC")) != length(trim(col("NOC"))) 
).show(truncate=False)

+--------+---------------+-------+--------------+
|Original|Original_Length|Trimmed|Trimmed_length|
+--------+---------------+-------+--------------+
+--------+---------------+-------+--------------+



In [0]:
athletes.select(
    col("Discipline").alias("Original"),
    length(col("Discipline")).alias("Original_length"),
    trim(col("Discipline")).alias("Trimmed"),
    length(trim(col("Discipline"))).alias("Trimmed_length")
).filter(
    length(col("Discipline")) != length(trim(col("Discipline")))
).show(
    truncate=False
)

+--------+---------------+-------+--------------+
|Original|Original_length|Trimmed|Trimmed_length|
+--------+---------------+-------+--------------+
+--------+---------------+-------+--------------+



**Rather than manually reaching out to every single column to verify issues. Created a function which will check all the columns for inconsistencies whenever a table name is passed to the function.**

In [0]:
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim, length

def check_spacing_issues(df, table_name="table"):
    string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
    
    issues_found = False
    for c in string_cols:
        flagged = df.select(
            col(c).alias("original"),
            length(col(c)).alias("original_length"),
            trim(col(c)).alias("trimmed"),
            length(trim(col(c))).alias("trimmed_length")
        ).filter(length(col(c)) != length(trim(col(c))))
        
        count = flagged.count()
        if count > 0:
            issues_found = True
            print(f"[{table_name}] Column '{c}': {count} row(s) with spacing issues")
            flagged.show(50, truncate=False)
    
    if not issues_found:
        print(f"[{table_name}] No spacing issues found in any string column.")

In [0]:
check_spacing_issues(athletes, "athletes")
check_spacing_issues(coaches, "coaches")
check_spacing_issues(teams, "teams")
check_spacing_issues(entriesgender, "entriesgender")
check_spacing_issues(medals, "medals")

[athletes] Column 'Name': 1 row(s) with spacing issues
+----------------+---------------+---------------+--------------+
|original        |original_length|trimmed        |trimmed_length|
+----------------+---------------+---------------+--------------+
|GIRAUD Aurelien |16             |GIRAUD Aurelien|15            |
+----------------+---------------+---------------+--------------+

[coaches] No spacing issues found in any string column.
[teams] No spacing issues found in any string column.
[entriesgender] No spacing issues found in any string column.
[medals] No spacing issues found in any string column.



- In table `Athletes`:
In column -> "Name" : Found inconsistent Spacing

- In table `Coaches`: NO ERRORS!
- In table `Teams`: NO ERRORS!
- In table `entriesgender`: NO ERRORS!
- In table `Medals`: NO ERRORS!

In [0]:
# Fixing the issue

athletes = athletes.withColumn("Name", trim(col("Name")))

In [0]:
# Checking if it's fixed

check_spacing_issues(athletes, "athletes")

[athletes] No spacing issues found in any string column.


### 2. Next checking for Data types — are numeric columns actually numeric

In [0]:
athletes.printSchema()
coaches.printSchema()
teams.printSchema()
entriesgender.printSchema()
medals.printSchema()

root
 |-- Name: string (nullable = true)
 |-- NOC: string (nullable = true)
 |-- Discipline: string (nullable = true)

root
 |-- Name: string (nullable = true)
 |-- NOC: string (nullable = true)
 |-- Discipline: string (nullable = true)
 |-- Event: string (nullable = true)

root
 |-- Name: string (nullable = true)
 |-- Discipline: string (nullable = true)
 |-- NOC: string (nullable = true)
 |-- Event: string (nullable = true)

root
 |-- Discipline: string (nullable = true)
 |-- Female: integer (nullable = true)
 |-- Male: integer (nullable = true)
 |-- Total: integer (nullable = true)

root
 |-- Rank: integer (nullable = true)
 |-- Team/NOC: string (nullable = true)
 |-- Gold: integer (nullable = true)
 |-- Silver: integer (nullable = true)
 |-- Bronze: integer (nullable = true)
 |-- Total: integer (nullable = true)
 |-- Rank by Total: integer (nullable = true)



### 3. Null / missing values

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when


def check_nulls(df, table_name="table"):
    null_counts = df.select(
        [spark_sum(when(col(c).isNull(), 1).otherwise(0)) for c in df.columns]
    )

    print(f"{table_name} null counts:")
    null_counts.show(truncate=False)

In [0]:
check_nulls(athletes, "athletes")
check_nulls(coaches, "coaches")
check_nulls(teams, "teams")
check_nulls(entriesgender, "entriesgender")
check_nulls(medals, "medals")

athletes null counts:
+-----------------------------------------------+----------------------------------------------+-----------------------------------------------------+
|sum(CASE WHEN (Name IS NULL) THEN 1 ELSE 0 END)|sum(CASE WHEN (NOC IS NULL) THEN 1 ELSE 0 END)|sum(CASE WHEN (Discipline IS NULL) THEN 1 ELSE 0 END)|
+-----------------------------------------------+----------------------------------------------+-----------------------------------------------------+
|0                                              |0                                             |0                                                    |
+-----------------------------------------------+----------------------------------------------+-----------------------------------------------------+

coaches null counts:
+-----------------------------------------------+----------------------------------------------+-----------------------------------------------------+------------------------------------------------+
|

In [0]:
coaches.where(col("Event").isNull()).show(10) # In Total 145 rows have NULL values

+--------------------+-------------+----------+-----+
|                Name|          NOC|Discipline|Event|
+--------------------+-------------+----------+-----+
|     ABDELMAGID Wael|        Egypt|  Football| NULL|
|           ABE Junya|        Japan|Volleyball| NULL|
|       ABE Katsuhiko|        Japan|Basketball| NULL|
|        ADAMA Cherif|Côte d'Ivoire|  Football| NULL|
|          AGEBA Yuya|        Japan|Volleyball| NULL|
|ALLER CARBALLO Ma...|        Spain|Basketball| NULL|
|           ALY Kamal|        Egypt|  Football| NULL|
| AMAYA GAITAN Fabian|  Puerto Rico|Basketball| NULL|
|    AMO AGUADO Pablo|        Spain|  Football| NULL|
|       BACIU Horatiu|      Romania|  Football| NULL|
+--------------------+-------------+----------+-----+
only showing top 10 rows


**As we can see, we got 145 NULL Values in Table: `Coaches`, Column: `Event`.**

We have to decide something about the NULL values either ask the producer or the team is there any value we can replace them with
For now replacing it with `N/A` (Not available) it's better for identification

In [0]:
# replacing Null values with N/A - Not Available

coaches = coaches.fillna({"Event":"N/A"})

In [0]:
coaches.filter(col("Event").isNull()).count()

0

Successfully removed Null Values from: Table - `Coaches`

- Total NULL value Count - 145

### 4. Duplicate rows

Table: `Athletes`

In [0]:
# Finding the difference b/w total count and count after dropping duplicates

print(athletes.count(), athletes.dropDuplicates().count())

11085 11084


In [0]:
# Fetching the duplicated rows

athletes.groupBy(athletes.columns).count().filter(col("count") > 1).show(truncate=False)

+-----------+-------+----------+-----+
|Name       |NOC    |Discipline|count|
+-----------+-------+----------+-----+
|ALI Mohamed|Bahrain|Handball  |2    |
+-----------+-------+----------+-----+



In [0]:
# removing duplicates

athletes = athletes.dropDuplicates()

In [0]:
print(athletes.count(), athletes.dropDuplicates().count())

11084 11084


In [0]:
def check_duplicates(df, table_name = "table"):
    total_count = df.count()
    distinct_count = df.dropDuplicates().count()
    duplicate_value = total_count - distinct_count

    if duplicate_value == 0:
        print(f"[{table_name}] No duplicate row(s) found. Duplicate Count: {duplicate_value}")
    else:
        print(f"[{table_name}] found duplicate row(s). Duplicate Count: {duplicate_value} & Total rows: {total_count}")
        df.groupBy(df.columns).count().filter(col("count") > 1).show(truncate=False)

In [0]:
check_duplicates(athletes, "athletes")
check_duplicates(coaches, "coaches")
check_duplicates(teams, "teams")
check_duplicates(entriesgender, "entriesgender")
check_duplicates(medals, "medals")

[athletes] No duplicate row(s) found. Duplicate Count: 0
[coaches] found duplicate row(s). Duplicate Count: 1 & Total rows: 394
+----------------+------+-----------------+--------+-----+
|Name            |NOC   |Discipline       |Event   |count|
+----------------+------+-----------------+--------+-----+
|GUERRERO Rolando|Mexico|Baseball/Softball|Softball|2    |
+----------------+------+-----------------+--------+-----+

[teams] No duplicate row(s) found. Duplicate Count: 0
[entriesgender] No duplicate row(s) found. Duplicate Count: 0
[medals] No duplicate row(s) found. Duplicate Count: 0


In [0]:
coaches = coaches.dropDuplicates()

In [0]:
check_duplicates(coaches, "coaches")

[coaches] No duplicate row(s) found. Duplicate Count: 0


### 5. Referential consistency between tables — the country/team-name mismatch problem I flagged earlier

This is the one most likely to bite you later. Check whether the same country is spelled identically across tables:

In [0]:
athletes.columns

['Name', 'NOC', 'Discipline']

In [0]:
medals.select("Team/NOC").distinct().show(100, truncate=False)
athletes.select("NOC").distinct().show(100, truncate=False)

+--------------------------+
|Team/NOC                  |
+--------------------------+
|Australia                 |
|Israel                    |
|Indonesia                 |
|Burkina Faso              |
|Côte d'Ivoire             |
|Ghana                     |
|Sweden                    |
|Croatia                   |
|Turkey                    |
|Romania                   |
|Thailand                  |
|Bermuda                   |
|Armenia                   |
|Denmark                   |
|Slovenia                  |
|Uzbekistan                |
|Slovakia                  |
|Tunisia                   |
|Kyrgyzstan                |
|Bahrain                   |
|North Macedonia           |
|Namibia                   |
|ROC                       |
|Switzerland               |
|Bahamas                   |
|Morocco                   |
|Lithuania                 |
|Syrian Arab Republic      |
|Hungary                   |
|Belarus                   |
|Austria                   |
|Japan        

In [0]:
medals = medals.withColumnRenamed("Team/NOC","Team_NOC")

In [0]:
medals.show(2)

+----+--------------------+----+------+------+-----+-------------+
|Rank|            Team_NOC|Gold|Silver|Bronze|Total|Rank by Total|
+----+--------------------+----+------+------+-----+-------------+
|   1|United States of ...|  39|    41|    33|  113|            1|
|   2|People's Republic...|  38|    32|    18|   88|            2|
+----+--------------------+----+------+------+-----+-------------+
only showing top 2 rows


In [0]:
athletes_countries = set(row["NOC"] for row in athletes.select("NOC").distinct().collect())
medals_countries = set(row["Team_NOC"] for row in medals.select("Team_NOC").distinct().collect())

only_in_athletes = athletes_countries - medals_countries
only_in_medals = medals_countries - athletes_countries

print("In athletes but not in medals:", only_in_athletes)
print("In medals but not in athletes:", only_in_medals)

In athletes but not in medals: {'Central African Republic', 'Virgin Islands, US', 'Montenegro', 'Federated States of Micronesia', 'Cape Verde', 'Cambodia', 'Myanmar', 'Sudan', 'Madagascar', 'Lesotho', 'Iraq', 'Sri Lanka', 'Niger', 'Paraguay', 'Djibouti', 'Chad', 'Peru', 'Democratic Republic of Timor-Leste', 'Benin', 'Uruguay', 'Luxembourg', 'Guinea-Bissau', 'Vietnam', 'Liberia', 'Gabon', 'Mauritania', 'Dominica', 'Maldives', 'Guyana', 'Eswatini', 'Togo', 'Pakistan', 'Equatorial Guinea', 'Comoros', 'Zimbabwe', 'Palestine', 'El Salvador', 'Sao Tome and Principe', 'United Arab Emirates', 'Mali', 'Angola', 'Seychelles', 'Yemen', 'Gambia', 'Honduras', 'Kiribati', 'Bangladesh', "Lao People's Democratic Republic", 'Panama', 'Singapore', 'Saint Lucia', 'St Vincent and the Grenadines', 'Brunei Darussalam', 'Sierra Leone', 'South Sudan', 'Nepal', 'Papua New Guinea', 'Senegal', 'Palau', 'Libya', 'Bosnia and Herzegovina', 'Trinidad and Tobago', 'Aruba', 'Suriname', 'Albania', 'Burundi', 'Afghanist

### 6. Value-range sanity checks (light validation)

Table: `medals`

In [0]:
medals.show(2)

+----+--------------------+----+------+------+-----+-------------+
|Rank|            Team_NOC|Gold|Silver|Bronze|Total|Rank by Total|
+----+--------------------+----+------+------+-----+-------------+
|   1|United States of ...|  39|    41|    33|  113|            1|
|   2|People's Republic...|  38|    32|    18|   88|            2|
+----+--------------------+----+------+------+-----+-------------+
only showing top 2 rows


In [0]:
medals.filter(
    (col("Gold") + col("Silver") + col("Bronze") != col("Total")) 
).show()

+----+--------+----+------+------+-----+-------------+
|Rank|Team_NOC|Gold|Silver|Bronze|Total|Rank by Total|
+----+--------+----+------+------+-----+-------------+
+----+--------+----+------+------+-----+-------------+



Table: `entriesgender`

In [0]:
entriesgender.columns

['Discipline', 'Female', 'Male', 'Total']

In [0]:
entriesgender.filter(
    col("Male") + col("Female") != col("Total")
).show()

+----------+------+----+-----+
|Discipline|Female|Male|Total|
+----------+------+----+-----+
+----------+------+----+-----+



### 7. Changing and confirming some column names to avoid future conflicts/confusions

In [0]:
medals.select(medals.columns).orderBy(col("Rank")).show()

+----+--------------------+----+------+------+-----+-------------+
|Rank|            Team_NOC|Gold|Silver|Bronze|Total|Rank by Total|
+----+--------------------+----+------+------+-----+-------------+
|   1|United States of ...|  39|    41|    33|  113|            1|
|   2|People's Republic...|  38|    32|    18|   88|            2|
|   3|               Japan|  27|    14|    17|   58|            5|
|   4|       Great Britain|  22|    21|    22|   65|            4|
|   5|                 ROC|  20|    28|    23|   71|            3|
|   6|           Australia|  17|     7|    22|   46|            6|
|   7|         Netherlands|  10|    12|    14|   36|            9|
|   8|              France|  10|    12|    11|   33|           10|
|   9|             Germany|  10|    11|    16|   37|            8|
|  10|               Italy|  10|    10|    20|   40|            7|
|  11|              Canada|   7|     6|    11|   24|           11|
|  12|              Brazil|   7|     6|     8|   21|          

In [0]:
medals = medals.withColumnsRenamed({
    "Rank":"Rank_by_gold",
    "Rank by Total":"Rank_by_total"
})
medals.show()

+------------+--------------------+----+------+------+-----+-------------+
|Rank_by_gold|            Team_NOC|Gold|Silver|Bronze|Total|Rank_by_total|
+------------+--------------------+----+------+------+-----+-------------+
|           1|United States of ...|  39|    41|    33|  113|            1|
|           2|People's Republic...|  38|    32|    18|   88|            2|
|           3|               Japan|  27|    14|    17|   58|            5|
|           4|       Great Britain|  22|    21|    22|   65|            4|
|           5|                 ROC|  20|    28|    23|   71|            3|
|           6|           Australia|  17|     7|    22|   46|            6|
|           7|         Netherlands|  10|    12|    14|   36|            9|
|           8|              France|  10|    12|    11|   33|           10|
|           9|             Germany|  10|    11|    16|   37|            8|
|          10|               Italy|  10|    10|    20|   40|            7|
|          11|           

# Loading the transformed data back to ADLS Gen 2

In [0]:

(
    athletes
    .repartition(1)
    .write
    .mode("overwrite")
    .parquet(
        "abfss://tokyo-olympic-data@tokyoolympicdatasohom.dfs.core.windows.net/transformed-data/athletes"
    )
)

(
    coaches
    .repartition(1)
    .write
    .mode("overwrite")
    .parquet(
        "abfss://tokyo-olympic-data@tokyoolympicdatasohom.dfs.core.windows.net/transformed-data/coaches"
    )
)

(
    teams
    .repartition(1)
    .write
    .mode("overwrite")
    .parquet(
        "abfss://tokyo-olympic-data@tokyoolympicdatasohom.dfs.core.windows.net/transformed-data/teams"
    )
)

(
    entriesgender
    .repartition(1)
    .write
    .mode("overwrite")
    .parquet(
        "abfss://tokyo-olympic-data@tokyoolympicdatasohom.dfs.core.windows.net/transformed-data/entriesgender"
    )
)

(
    medals
    .repartition(1)
    .write
    .mode("overwrite")
    .parquet(
        "abfss://tokyo-olympic-data@tokyoolympicdatasohom.dfs.core.windows.net/transformed-data/medals"
    )
)

In [0]:
display(dbutils.fs.ls("abfss://tokyo-olympic-data@tokyoolympicdatasohom.dfs.core.windows.net/transformed-data/"))

path,name,size,modificationTime
abfss://tokyo-olympic-data@tokyoolympicdatasohom.dfs.core.windows.net/transformed-data/athletes/,athletes/,0,1790020248000
abfss://tokyo-olympic-data@tokyoolympicdatasohom.dfs.core.windows.net/transformed-data/coaches/,coaches/,0,1790020249000
abfss://tokyo-olympic-data@tokyoolympicdatasohom.dfs.core.windows.net/transformed-data/entriesgender/,entriesgender/,0,1790020251000
abfss://tokyo-olympic-data@tokyoolympicdatasohom.dfs.core.windows.net/transformed-data/medals/,medals/,0,1790020252000
abfss://tokyo-olympic-data@tokyoolympicdatasohom.dfs.core.windows.net/transformed-data/teams/,teams/,0,1790020250000


In [0]:
# quick sanity check

spark.read.parquet("abfss://tokyo-olympic-data@tokyoolympicdatasohom.dfs.core.windows.net/transformed-data/medals").printSchema()

root
 |-- Rank_by_gold: integer (nullable = true)
 |-- Team_NOC: string (nullable = true)
 |-- Gold: integer (nullable = true)
 |-- Silver: integer (nullable = true)
 |-- Bronze: integer (nullable = true)
 |-- Total: integer (nullable = true)
 |-- Rank_by_total: integer (nullable = true)

